# ML Assignment - Spam Classification
**Student:** Shaked Sabag. | **Last 4 ID digits:** 2142  
**Dataset:** SMS Spam Collection (Kaggle)  
**Algorithm:** Naive Bayes Classifier  
**Task:** Binary Text Classification (Spam / Ham)

## AI Tools Used
- Perplexity AI: "explain TF-IDF vectorization for spam detection"
- Perplexity AI: "how to implement Naive Bayes classifier with sklearn"

## Dataset Description
The SMS Spam Collection dataset contains 5,572 SMS messages labeled as either
'spam' or 'ham' (legitimate). The dataset is imbalanced: ~87% ham, ~13% spam.
The goal is to build a binary classifier that automatically detects spam messages
based on their text content using NLP feature extraction techniques.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import f1_score, classification_report, confusion_matrix

In [ ]:
# טעינת הקובץ
df = pd.read_csv('spam_data.csv')
print("Total rows:", len(df))
print(df['Category'].value_counts())
df.head(5)

In [ ]:
# חלוקה ל-train/test — פעם אחת בלבד!
train_df, test_df = train_test_split(
    df,
    test_size=0.2,        # 80% train, 20% test
    random_state=42,      # חשוב! כדי שיהיה reproducible
    stratify=df['Category']  # שומר על יחס spam/ham בשני החלקים
)

print("Train size:", len(train_df))
print("Test size:", len(test_df))

# שמור לקבצים
train_df.to_csv('train.csv', index=False)
test_df.to_csv('test.csv', index=False)

print("Done! train.csv and test.csv saved.")

## Part 2 - Feature Engineering
We experiment with two vectorization methods learned in class:
1. **TF-IDF** (Term Frequency - Inverse Document Frequency)
2. **Bag of Words** (CountVectorizer)

Each method converts raw text into numerical vectors that the ML algorithm can process.

In [ ]:
# שיטה 1: TF-IDF
tfidf = TfidfVectorizer(max_features=1000, stop_words='english')
X_train_tfidf = tfidf.fit_transform(train_df['Message'])
X_test_tfidf  = tfidf.transform(test_df['Message'])

print("TF-IDF matrix shape (train):", X_train_tfidf.shape)

# הצגת 3 דוגמאות מה-train
print("\n--- 3 Train Examples (TF-IDF) ---")
sample_df = pd.DataFrame(
    X_train_tfidf[:3].toarray(),
    columns=tfidf.get_feature_names_out()
)
sample_df.iloc[:, :8]  # 8 עמודות ראשונות

In [ ]:
# שיטה 2: Bag of Words
bow = CountVectorizer(max_features=1000, stop_words='english')
X_train_bow = bow.fit_transform(train_df['Message'])
X_test_bow   = bow.transform(test_df['Message'])

print("BoW matrix shape (train):", X_train_bow.shape)

# הצגת 3 דוגמאות מה-test
print("\n--- 3 Test Examples (BoW) ---")
sample_test = pd.DataFrame(
    X_test_bow[:3].toarray(),
    columns=bow.get_feature_names_out()
)
sample_test.iloc[:, :8]

## Part 3 - Learning Algorithm: Naive Bayes

**Naive Bayes** applies Bayes' theorem with the "naive" assumption that features
(words) are conditionally independent given the class.

**Formula:** P(spam | words) ∝ P(words | spam) × P(spam)

- **Prior** P(spam): % of spam messages in training data
- **Likelihood** P(word | spam): how often each word appears in spam
- **Alpha (smoothing)**: prevents zero probability for unseen words (Laplace smoothing)

In [ ]:
class SpamClassifier:
    """
    Naive Bayes text classifier.
    Supports hyperparameter tuning via alpha (Laplace smoothing).
    """
    def __init__(self, alpha=1.0):
        self.alpha = alpha
        self.model = MultinomialNB(alpha=self.alpha)

    def train(self, X, y):
        self.model.fit(X, y)
        print(f"✅ Model trained | alpha={self.alpha}")

    def predict(self, X):
        return self.model.predict(X)

    def predict_proba(self, X):
        return self.model.predict_proba(X)

# אימון על TF-IDF
clf = SpamClassifier(alpha=1.0)
clf.train(X_train_tfidf, train_df['Category'])

## Part 4 - Training Flow
Below we demonstrate 3 examples passing through the full pipeline:
raw text → TF-IDF vectorization → classifier → prediction

In [ ]:
examples = train_df['Message'].iloc[:3].tolist()
labels   = train_df['Category'].iloc[:3].tolist()

print("=== Training Flow — 3 Examples ===\n")
for i, (msg, label) in enumerate(zip(examples, labels)):
    vec  = tfidf.transform([msg])
    pred = clf.predict(vec)[0]
    prob = clf.predict_proba(vec)[0].max()
    status = "✅" if pred == label else "❌"
    print(f"[{i+1}] Text: '{msg[:60]}...'")
    print(f"     Real: {label} | Predicted: {pred} | Confidence: {prob:.2%} {status}\n")

## Part 5 - Model Evaluation on Test Set
We use **F1 Score** (binary) as the primary metric since this is a binary
classification task with an imbalanced dataset (spam is the positive class).

In [ ]:
y_pred = clf.predict(X_test_tfidf)
y_true = test_df['Category']

# 5 חיזויים ראשונים
print("=== First 5 Predictions ===")
for i in range(5):
    print(f"Real: {y_true.iloc[i]:4} | Predicted: {y_pred[i]:4} | '{test_df['Message'].iloc[i][:50]}'")

# מדדי איכות
print("\n=== Model Quality ===")
print(f"F1 Score (spam): {f1_score(y_true, y_pred, pos_label='spam'):.4f}")
print(classification_report(y_true, y_pred))

# Confusion Matrix
cm = confusion_matrix(y_true, y_pred, labels=['spam', 'ham'])
plt.figure(figsize=(6, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['spam','ham'], yticklabels=['spam','ham'])
plt.title('Confusion Matrix')
plt.ylabel('Actual')
plt.xlabel('Predicted')
plt.tight_layout()
plt.savefig('confusion_matrix.png')
plt.show()

## Part 6 - Bonus: Grid Search + K-Fold Cross Validation
Testing all combinations of feature engineering methods and hyperparameters
using 5-Fold Cross Validation to find the best model configuration.

In [ ]:
from sklearn.model_selection import GridSearchCV, cross_val_score

results = []

for feat_name, X_tr, X_te in [
    ("TF-IDF", X_train_tfidf, X_test_tfidf),
    ("BoW",    X_train_bow,   X_test_bow),
]:
    for alpha in [0.1, 0.5, 1.0, 2.0]:
        model = MultinomialNB(alpha=alpha)
        scores = cross_val_score(
            model, X_tr, train_df['Category'],
            cv=5, scoring='f1_macro'
        )
        results.append({
            'Feature':    feat_name,
            'Alpha':      alpha,
            'CV F1 Mean': round(scores.mean(), 4),
            'CV F1 Std':  round(scores.std(), 4),
        })

results_df = pd.DataFrame(results).sort_values('CV F1 Mean', ascending=False)
print("=== All Permutations ===")
print(results_df.to_string(index=False))

best = results_df.iloc[0]
print(f"\n🏆 Best: Feature={best['Feature']} | Alpha={best['Alpha']} | CV F1={best['CV F1 Mean']}")

In [ ]:
# אמן מחדש על כל ה-trainset עם הפרמוטציה המנצחת
best_feat  = best['Feature']
best_alpha = best['Alpha']

X_final_train = X_train_tfidf if best_feat == "TF-IDF" else X_train_bow
X_final_test  = X_test_tfidf  if best_feat == "TF-IDF" else X_test_bow

final_clf = SpamClassifier(alpha=best_alpha)
final_clf.train(X_final_train, train_df['Category'])

# הערכה סופית
y_final_pred = final_clf.predict(X_final_test)
final_f1 = f1_score(test_df['Category'], y_final_pred, pos_label='spam')
print(f"✅ Final Test F1 Score: {final_f1:.4f}")

In [ ]:
import shap

vectorizer = tfidf if best_feat == "TF-IDF" else bow

# עבור Naive Bayes משתמשים ב-KernelExplainer דרך predict_proba
# לוקחים sample קטן (50) כי KernelExplainer איטי יותר
background = X_final_train[:50]
test_sample = X_final_test[:20]

explainer = shap.KernelExplainer(
    final_clf.model.predict_proba,
    background
)
shap_values = explainer.shap_values(test_sample, nsamples=100)

# shap_values[1] = class "spam"
shap.summary_plot(
    shap_values[1],
    feature_names=vectorizer.get_feature_names_out(),
    max_display=15,
    show=True
)